# XGBoost (with hyperparameter search)

Gradient-boosted trees on the shared engineered features (Store, Dept, calendar,
markdowns, economic signals). We search a few hyperparameter configs, log each as its
own MLflow run, keep the best by validation WMAE, refit it on all data, and register it.

WMAE uses the shared `calculate_wmae` on the same validation split as the other models.
Logged to the shared DagsHub.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import yaml
import warnings
import pandas as pd
import mlflow
import mlflow.sklearn
import dagshub

from src.features.preprocessing import get_model_ready_data
from src.models.xgboost_pipeline import create_xgboost_pipeline
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings('ignore')
dagshub.init(repo_owner='slosa23', repo_name='Walmart-Sales-Forecasting', mlflow=True)
mlflow.set_experiment('XGBoost_HyperSearch')

with open('configs/xgboost_config.yaml') as f:
    config = yaml.safe_load(f)
COMMON = config['model']['common']
SEARCH_SPACE = config['model']['search_space']
split_date = config['data']['split_date']
print('candidates:', len(SEARCH_SPACE))

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as GigiSichinava

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

2026/07/12 20:15:14 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_HyperSearch' does not exist. Creating a new experiment.


candidates: 5


In [3]:
train_raw = pd.read_csv('data/train.csv')
stores = pd.read_csv('data/stores.csv')
features = pd.read_csv('data/features.csv')

X_train, y_train, X_val, y_val, is_holiday_val, preprocessor = get_model_ready_data(
    train_raw, stores, features, split_date=split_date
)
print('train:', X_train.shape, '| val:', X_val.shape)

train: (294132, 17) | val: (127438, 17)


## Search the hyperparameter candidates

Each candidate merges `common` with its own overrides, trains on the training period,
and is scored on the validation rows. The best-WMAE config is kept.

In [4]:
best_wmae, best_params = float('inf'), None
for i, cand in enumerate(SEARCH_SPACE, 1):
    params = {**COMMON, **cand}
    run_name = f"XGB_cfg{i}_est{cand['n_estimators']}_d{cand['max_depth']}_lr{cand['learning_rate']}"
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        pipe = create_xgboost_pipeline(preprocessor, params)
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_val)
        wmae = calculate_wmae(y_val, preds, is_holiday_val)
        mlflow.log_metric('WMAE', float(wmae))
        print(f'{run_name}: WMAE {wmae:,.2f}')
        if wmae < best_wmae:
            best_wmae, best_params = wmae, params
print('---')
print('Best WMAE:', round(best_wmae, 2))
print('Best params:', best_params)

XGB_cfg1_est250_d7_lr0.05: WMAE 3,327.24
🏃 View run XGB_cfg1_est250_d7_lr0.05 at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/13/runs/00060fe927ff46b88ff27ea9b866cbbc
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/13
XGB_cfg2_est500_d8_lr0.03: WMAE 2,840.41
🏃 View run XGB_cfg2_est500_d8_lr0.03 at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/13/runs/787fea2f1b9a43c5878f66fefcd5613a
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/13
XGB_cfg3_est400_d6_lr0.05: WMAE 3,674.60
🏃 View run XGB_cfg3_est400_d6_lr0.05 at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/13/runs/e82601a29b94426f882a35753cd26132
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/13
XGB_cfg4_est600_d10_lr0.02: WMAE 2,368.40
🏃 View run XGB_cfg4_est600_d10_lr0.02 at: https://dagshub.com/sl

## Refit the best config on all data and register it

In [5]:
with mlflow.start_run(run_name='XGBoost_Best_Registered'):
    mlflow.log_params(best_params)
    mlflow.log_metric('val_WMAE', float(best_wmae))

    X_all = pd.concat([X_train, X_val], axis=0)
    y_all = pd.concat([y_train, y_val], axis=0)
    best_pipe = create_xgboost_pipeline(preprocessor, best_params)
    best_pipe.fit(X_all, y_all)

    trusted_types = [
        'sklearn.compose._column_transformer._RemainderColsList',
        'xgboost.core.Booster',
        'xgboost.sklearn.XGBRegressor',
        'numpy.dtype',
    ]
    mlflow.sklearn.log_model(
        sk_model=best_pipe,
        name='model',
        skops_trusted_types=trusted_types,
        registered_model_name='Walmart_XGBoost_Baseline',
    )
    print('Registered best XGBoost (fit on all data). Val WMAE:', round(best_wmae, 2))

Registered model 'Walmart_XGBoost_Baseline' already exists. Creating a new version of this model...
2026/07/12 20:21:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_XGBoost_Baseline, version 3
Created version '3' of model 'Walmart_XGBoost_Baseline'.


Registered best XGBoost (fit on all data). Val WMAE: 2000.87
🏃 View run XGBoost_Best_Registered at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/13/runs/294c2476098948959b8500c2514ea162
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/13


## Notes

- Search space lives in `configs/xgboost_config.yaml`; the pipeline builder is in
  `src/models/xgboost_pipeline.py` (unchanged).
- WMAE on the same validation split as the other models, via shared `calculate_wmae`.
- The best config is refit on all data and registered as `Walmart_XGBoost_Baseline`.